In [1]:
import sys
sys.path.insert(0, ".")

from config import MODELS, BFCL_CATEGORIES
from data.loader import load_bfcl_category, load_ground_truth
from data.preprocessor import bfcl_to_openai_tools
from inference.runner import run_single, run_category
from evaluation.metrics import evaluate_results, compute_category_metrics, CategoryMetrics
from analysis.report import build_accuracy_table, build_token_efficiency_table, build_version_delta_table
from analysis.charts import plot_accuracy_by_category, plot_token_efficiency, plot_latency_comparison
from tqdm.notebook import tqdm
import pandas as pd

pd.set_option("display.max_colwidth", None)
print("All imports OK")
print(f"Models: {[m['label'] for m in MODELS]}")
print(f"Categories: {BFCL_CATEGORIES}")

All imports OK
Models: ['Granite 4.1 8B', 'Granite 4.0', 'Llama 3.1 8B', 'Qwen 2.5 7B', 'Mistral 7B']
Categories: ['simple', 'multiple', 'parallel', 'parallel_multiple']


In [2]:
# Load all results from cache (skips the multi-hour inference run)
import json
from pathlib import Path

all_raw_results = {}
for model in MODELS:
    tag, label = model["tag"], model["label"]
    safe_tag = tag.replace(":", "_").replace("/", "_")
    all_raw_results[label] = {}
    for cat in BFCL_CATEGORIES:
        cat_dir = Path("results") / safe_tag
        files = sorted(cat_dir.glob(f"{cat}_*.json"))
        all_raw_results[label][cat] = [json.loads(f.read_text()) for f in files]
    total = sum(len(v) for v in all_raw_results[label].values())
    print(f"{label}: {total} results loaded")
print("Cache loaded. Ready for evaluation.")

Granite 4.1 8B: 1200 results loaded


Granite 4.0: 1200 results loaded


Llama 3.1 8B: 1200 results loaded


Qwen 2.5 7B: 1200 results loaded


Mistral 7B: 1200 results loaded
Cache loaded. Ready for evaluation.


In [ ]:
# Verify full pipeline works on 2 samples before the multi-hour run
samples = load_bfcl_category("simple")[:2]
gt = load_ground_truth("simple")

test_model = MODELS[0]["tag"]  # granite4.1:8b
print(f"Smoke testing: {test_model}\n")
for s in samples:
    r = run_single(test_model, s)
    evaluated = evaluate_results([r], gt, "simple")
    e = evaluated[0]
    print(f"ID: {r['id']} | tool_calls: {r['tool_calls']}")
    print(f"  tokens: {r['completion_tokens']} | latency: {r['latency_ms']:.0f}ms")
    print(f"  name_correct={e['name_correct']} full_match={e['full_match']}\n")

In [ ]:
# Full benchmark — ~3-7 hours. Caches per-call to results/. Fully resumable.
all_raw_results = {}  # {model_label: {category: [result dicts]}}

for model in MODELS:
    tag, label = model["tag"], model["label"]
    print(f"\n{'='*50}\nRunning: {label} ({tag})\n{'='*50}")
    all_raw_results[label] = {}

    for cat in BFCL_CATEGORIES:
        samples = load_bfcl_category(cat)
        with tqdm(total=len(samples), desc=f"  {cat}") as pbar:
            results = run_category(tag, samples, progress=pbar)
        all_raw_results[label][cat] = results
        cached = sum(1 for s in samples
                     if (MODELS[0]["tag"] != tag or True)  # always show
                     )
        print(f"  {cat}: {len(results)} calls done")

print("\nAll inference complete.")

In [5]:
all_metrics = {}  # {model_label: {category: CategoryMetrics}}

for model in MODELS:
    label = model["label"]
    all_metrics[label] = {}
    for cat in BFCL_CATEGORIES:
        gt = load_ground_truth(cat)
        raw = all_raw_results[label][cat]
        evaluated = evaluate_results(raw, gt, cat)
        all_metrics[label][cat] = compute_category_metrics(evaluated)

print("Evaluation complete.")

Evaluation complete.


In [6]:
acc_table = build_accuracy_table(all_metrics)
print("=== Full AST Accuracy by Category ===")
display(acc_table)

=== Full AST Accuracy by Category ===


,Simple,Multiple,Parallel,Parallel Multiple,Overall
Model,,,,,
Granite 4.1 8B,50.2%,46.5%,30.0%,47.0%,42.2%
Granite 4.0,43.0%,39.5%,24.2%,34.5%,34.8%
Llama 3.1 8B,49.5%,51.5%,24.0%,44.0%,40.4%
Qwen 2.5 7B,53.0%,48.0%,28.0%,47.0%,42.8%
Mistral 7B,48.2%,41.0%,21.2%,29.5%,34.9%


In [7]:
tok_table = build_token_efficiency_table(all_metrics)
print("=== Token Efficiency (fewer = better) ===")
display(tok_table)

=== Token Efficiency (fewer = better) ===


,Avg Tokens (Correct Calls)
Model,
Llama 3.1 8B,56.7
Granite 4.0,62.1
Granite 4.1 8B,63.5
Qwen 2.5 7B,64.3
Mistral 7B,104.8


In [8]:
delta_table = build_version_delta_table(
    all_metrics,
    model_new="Granite 4.1 8B",
    model_old="Granite 4.0",
)
print("=== Granite 4.1 vs 4.0 — Accuracy Delta (percentage points) ===")
display(delta_table)

=== Granite 4.1 vs 4.0 — Accuracy Delta (percentage points) ===


,Granite 4.1 8B Acc %,Granite 4.0 Acc %,Delta (pp)
Category,,,
Simple,50.2%,43.0%,+7.2
Multiple,46.5%,39.5%,+7.0
Parallel,30.0%,24.2%,+5.8
Parallel Multiple,47.0%,34.5%,+12.5


In [9]:
import matplotlib
matplotlib.use("Agg")

plot_accuracy_by_category(all_metrics)
plot_token_efficiency(all_metrics)

flat_raw = {
    label: [r for cat_results in all_raw_results[label].values() for r in cat_results]
    for label in all_raw_results
}
plot_latency_comparison(flat_raw)

Saved: results/accuracy_by_category.png
Saved: results/token_efficiency.png


Saved: results/latency_comparison.png


In [10]:
print("=== SUMMARY ===\n")
for model in MODELS:
    label = model["label"]
    total_c = sum(int(m.full_acc * m.total) for m in all_metrics[label].values())
    total_n = sum(m.total for m in all_metrics[label].values())
    overall_acc = total_c / total_n if total_n > 0 else 0

    all_correct_tokens = []
    for m in all_metrics[label].values():
        if m.avg_tokens_correct > 0:
            all_correct_tokens.extend([m.avg_tokens_correct] * max(1, int(m.full_acc * m.total)))
    avg_tok = sum(all_correct_tokens) / len(all_correct_tokens) if all_correct_tokens else 0

    print(f"{label:20s}  Overall Acc: {overall_acc*100:5.1f}%  Avg Tokens (correct): {avg_tok:5.1f}")

print("\nCalibration check: IBM published Granite 4.1 8B BFCL v3 score = 68.27%")
print("Your result should be within ±5pp of that to confirm correct setup.")

=== SUMMARY ===

Granite 4.1 8B        Overall Acc:  42.2%  Avg Tokens (correct):  63.5
Granite 4.0           Overall Acc:  34.8%  Avg Tokens (correct):  62.1
Llama 3.1 8B          Overall Acc:  40.4%  Avg Tokens (correct):  56.7
Qwen 2.5 7B           Overall Acc:  42.8%  Avg Tokens (correct):  64.3
Mistral 7B            Overall Acc:  34.9%  Avg Tokens (correct): 104.8

Calibration check: IBM published Granite 4.1 8B BFCL v3 score = 68.27%
Your result should be within ±5pp of that to confirm correct setup.
